# Completed Experiment Comparison

Read-only presentation of historical S20/640 v0/v1/v2, the merged PR #52 Raw S20/960 reference, and the current v3 adaptive research decision. An unconfirmed/untrained v3 receives no trained-development metric row.

**Inputs:** Committed per-study evidence under docs/issue45-s20, docs/issue47, docs/issue47-s20-resolution-20e and docs/issue47-v3-960.

**Outputs:** In-memory displays and charts only. No local runs, weights or datasets.

**Prerequisites:** Committed compact evidence and the project notebook kernel. No training, inference, pixel access, resampling or upstream notebook execution.

<a id="section-1"></a>

## 1. Configuration

All inputs come from committed compact evidence. This notebook has no execution opt-ins and does not call the historical evidence-acquisition implementation.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from vehicle_detection.common import file_hash, read_json, require_paths
from vehicle_detection.five_class import FIVE_CLASSES

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
REPORT = ROOT / "docs/issue45-s20"
EXPERIMENTS = {"v0": "Raw", "v1": "Gamma-or-Skip", "v2": "Local-shadow (exploratory)"}
require_paths(summary=REPORT / "summary.json", evidence=REPORT / "evidence.json")
for name, expected in read_json(REPORT / "evidence.json").items():
    if Path(name).name != name or file_hash(REPORT / name) != expected:
        raise ValueError("Committed comparison evidence changed")
summary = read_json(REPORT / "summary.json")
assert summary["training_subset"] == "S20" and summary["seed"] == 2026
assert summary["role"] == "development_validation" and not summary["final_test_accessed"]
records = summary["records"]
metrics = {version: read_json(REPORT / f"{version}_metrics.json") for version in EXPERIMENTS}
for version, result in metrics.items():
    assert result == records[version]["native"]
    assert result["inputs"]["training_subset"] == "S20"
    assert result["role"] == "development_validation" and not result["final_test_accessed"]

V3_REPORT = ROOT / "docs/issue47"
require_paths(v3_evidence=V3_REPORT / "evidence.json")
for name, expected in read_json(V3_REPORT / "evidence.json").items():
    if Path(name).name != name or file_hash(V3_REPORT / name) != expected:
        raise ValueError("Committed v3 evidence changed")
v3 = read_json(V3_REPORT / "summary.json")
assert not v3["final_test_accessed"] and not v3["development_accessed"]

RAW960_REPORT = ROOT / "docs/issue47-s20-resolution-20e"
CURRENT_V3 = ROOT / "docs/issue47-v3-960"
for directory in (RAW960_REPORT, CURRENT_V3):
    require_paths(evidence=directory / "evidence.json")
    for name, expected in read_json(directory / "evidence.json").items():
        if Path(name).name != name or file_hash(directory / name) != expected:
            raise ValueError("Committed 960 study evidence changed")
raw960 = read_json(RAW960_REPORT / "result.json")
raw960_checkpoint = read_json(RAW960_REPORT / "train960_checkpoint.json")
current_v3 = read_json(CURRENT_V3 / "result.json")
assert raw960["development_images"] == 13975
assert current_v3["checkpoint_sha256"] == raw960_checkpoint["checkpoint_hashes"]["best"]
assert not current_v3["final_test_accessed"] and not current_v3["development_accessed"]

<a id="section-2"></a>

## 2. Historical S20/640 Development Results

Same 9,781 training IDs, 13,975 development IDs and original five-class GT/negatives; independently initialized from the same pinned checkpoint, seed 2026, 20 epochs, 640, batch 16. Checkpoints were selected on shared v0 Raw development, then evaluated on each matching policy.

Percentages below are native dataset-level AP. CPU mean/p95 measurements use the same 256 consumed exploration images, eight threads, three repeats and rotated order. They include native quality diagnostics and exclude reads/JPEG/inference/verification equally; they are not deployment latency.


In [ ]:
overall = pd.DataFrame(
    [
        {
            "Version": version,
            "Policy": policy,
            "mAP50 (%)": metrics[version]["overall"]["mAP50"] * 100,
            "mAP50:95 (%)": metrics[version]["overall"]["mAP50_95"] * 100,
            "Selected epoch": records[version]["selected_epoch"],
            "CPU mean (ms)": records[version]["runtime"]["mean_ms"],
            "CPU p95 (ms)": records[version]["runtime"]["p95_ms"],
        }
        for version, policy in EXPERIMENTS.items()
    ]
).set_index("Version")
display(overall.round(4))
classes = pd.DataFrame(
    [
        {
            "Version": version,
            "Class": row["Class"],
            "AP50 (%)": row["mAP50"] * 100,
            "AP50:95 (%)": row["mAP50-95"] * 100,
        }
        for version, result in metrics.items()
        for row in result["classes"]
    ]
)
display(
    classes.pivot(index="Class", columns="Version", values=["AP50 (%)", "AP50:95 (%)"])
    .reindex(FIVE_CLASSES)
    .round(4)
)
display(Image(filename=str(REPORT / "comparison.png")))

<a id="section-3"></a>

## 3. Subgroups and Errors

Subgroups use frozen definitions: original brightness <0.12 (dark), >=0.12 (bright), sharpness <50 (pixel-defined low sharpness), and images containing motorcycle/bicycle boxes smaller than 32² original pixels. Every GT in a subgroup image remains included; these are image subsets, not size-specific AP-small. Subgroups overlap except dark/bright.

The low-sharpness subset has no bicycle GT and only one bus/motorcycle source group each. Its macro covers four observed classes and rare-class effects are inconclusive. Dark average AP rises slightly for v2, but dark motorcycle AP falls. Do not infer universal night/blur/small-target benefit from these averages.


In [ ]:
support = read_json(REPORT / "support.json")
subgroups = ("dark", "bright", "blurred", "small_two_wheeler")
display(
    pd.DataFrame(
        [
            {
                "Subgroup": group,
                "Images": support[group]["images"],
                "Represented classes": len(records["v0"]["subgroups"][group]["classes"]),
                **{
                    version: records[version]["subgroups"][group]["mAP50_95"] * 100
                    for version in EXPERIMENTS
                },
            }
            for group in subgroups
        ]
    )
    .set_index("Subgroup")
    .round(4)
)
print("Subgroup values are AP50:95 percentages, not per-image AP.")
display(
    pd.DataFrame({group: support[group]["class_source_groups"] for group in subgroups}).reindex(
        FIVE_CLASSES
    )
)
print("Native counts at confidence .25 / IoU .50; they do not replace ranked/localized AP.")
display(pd.DataFrame(summary["errors"]["all"]).T)
display(Image(filename=str(REPORT / "errors.png")))

<a id="section-4"></a>

## 4. Separate Historical Evidence

- **Historical S10 trained development:** 4,890 training images; [original report](../../docs/development-five-v1/RESULTS.md). These measurements are not S20 rows above.
- **Pretrained local-shadow pilot:** 256 calibration-exploration images and the frozen pretrained detector; [failed pilot report](../../docs/ISSUE45_RESULTS.md). Its gate failures and hash-bound-but-not-separately-committed declaration remain unchanged. It is not a trained S20 result.
- **S20 execution provenance:** [completed lifecycle and original diagnostic acquisition](../../docs/ISSUE45_S20_EXECUTION.md). Historical commands there describe how evidence was acquired; this notebook only reads that evidence.

The pilot and S20 populations differ, so comparing their effect sizes cannot isolate a causal fine-tuning effect.


## 5. Raw S20/960 Reference from Merged PR #52

The verified 9,781-image/20-epoch Raw960 detector is the reference for the new v3 study. This section reads its own saved 13,975-image development evidence, without replaying training or inference. The checkpoint was selected at epoch 10 using actual-640 Raw development / pad0, then evaluated at nominal960 / native pad.5. Historical S20/640 above used its historical selection behavior; these rows differ in resolution and selection details and are not a matched preprocessing contrast.

PR #52's resolution-training assessment remains inconclusive because dark-bus FN increased with an AP interval spanning zero. Reusing its checkpoint as a reference does not promote that earlier hypothesis or constitute independent confirmation.

In [ ]:
reference = raw960["metrics"]["960/960"]["all"]
display(
    {
        "Population": "S20 Raw / development",
        "Training images": 9781,
        "Development images": 13975,
        "Train/inference resolution": "960/960",
        "Selected epoch": raw960_checkpoint["selected_epoch"],
        "mAP50 (%)": reference["mAP50"] * 100,
        "mAP50:95 (%)": reference["mAP50_95"] * 100,
        "Checkpoint SHA256": raw960_checkpoint["checkpoint_hashes"]["best"],
    }
)
display(pd.DataFrame(reference["classes"]).T)
display(raw960["decision"])

## 6. Current V3 Adaptive Lifecycle

The historical pretrained Gamma/Sharpen pilot is preserved under docs/issue47; its original failed gates remain failed. The current bounded luminance/local-contrast study uses the **trained Raw S20/960** detector on **1,479 fresh calibration images**, never development. Raw, always-on and adaptive arms are paired on those same source IDs/GT; their AP belongs to this calibration population and cannot become a trained v3 development row.

The main candidate alone may advance. Only a frozen policy passing one untouched, group-disjoint confirmation can authorize S20 preparation and independent training. The lifecycle notebooks report skipped stages when that prerequisite fails. Complete intervals/draws, per-class/cohort effects, route-control tests and CPU costs are in the current study's own committed evidence.

In [ ]:
print("Historical pretrained pilot:", v3["population"])
display(v3["decision"])
print("Current v3:", current_v3["population"])
display(current_v3["decision"])
display(current_v3["routing"])
display(pd.DataFrame(current_v3["runtime"]).T)
print("Primary paired interval:")
display(current_v3["intervals"]["all"])
print("Policy frozen:", current_v3["policy_frozen"], "; v3 trained:", current_v3["v3_trained"])
print("No trained v3 development AP exists unless both confirmation and training complete.")

## 7. Comparison Summary

Historical v0/v1/v2 results and their conclusions remain unchanged. The Raw S20/960 reference comes directly from PR #52's verified saved evidence. Current v3 calibration evidence has its own prospective gates and explicit status; no untrained v3 result is inserted into the development comparison.

All trained reference results are single-seed reused-development estimates. The new calibration study does not create a final-test claim. Final-test pixels remain sealed; no S100 or v4 is introduced.